# Baseline: LinearProjector (300d - 3d)

In [19]:
# %pip install gensim

In [27]:
import sys
sys.path.insert(0, '..')
import torch, os
from gensim.models import KeyedVectors
from src.data import load_data
from src.models import build_model
from src.train import train
from src.evaluate import run_all_evaluations
from src.visualize import plot_curves, visualize_3d_projection
os.makedirs('../results/linear', exist_ok=True) 

In [28]:
from gensim.models import KeyedVectors

embeddings = KeyedVectors.load_word2vec_format(
    '../gensim-data/glove-wiki-gigaword-300/glove-wiki-gigaword-300.gz'
)
print(f'Vocab size: {len(embeddings.key_to_index)}')

Vocab size: 400000


In [29]:
train_loader, val_loader, gender_clean, royalty_clean = load_data(embeddings) 


--------------------------------------------------
  GENDER PAIRS
--------------------------------------------------
  candidates: 100 | valid ones: 100 | discarded: 0

--------------------------------------------------
  ROYALTY PAIRS
--------------------------------------------------
  candidates: 106 | valid ones: 106 | discarded: 0

  analysis: GENDER
  cos mean=0.4609 std=0.1898 | threshold 0.3: 83/100

  analysis: STATUS
  cos mean=0.2943 std=0.1196 | threshold 0.3: 51/106

  Cosine between axes: 0.3172

  Train: 106 | Val: 28


In [30]:
model = build_model('linear')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model, history = train(
    projector=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, lambda_ortho=0.1, num_epochs=1500,
)
torch.save(model.state_dict(), '../results/linear/model.pt')

Training:   0%|          | 0/1500 [00:00<?, ?it/s]


Best model: epoch 1168, val_loss=0.0614


In [31]:
plot_curves(history)

In [32]:
metrics = run_all_evaluations(model, embeddings)
print(metrics) 

=== Аналогии ===
  king - man + woman → queen  |  cos=0.9887  dist=0.2921
  prince - boy + girl → princess  |  cos=0.9819  dist=0.9253
  husband - man + woman → wife  |  cos=0.9585  dist=1.1758
  uncle - man + woman → aunt  |  cos=0.9826  dist=0.2453
  father - man + woman → mother  |  cos=0.9999  dist=0.0984
  brother - man + woman → sister  |  cos=0.9930  dist=0.3289
  king - prince + princess → queen  |  cos=0.9890  dist=0.5588
  grandfather - father + mother → grandmother  |  cos=0.9992  dist=0.1803
  actor - man + woman → actress  |  cos=0.9900  dist=0.6933
  he - man + woman → she  |  cos=0.9937  dist=0.2707
  analogy_cos=0.9877 | analogy_dist=0.4769
{'analogy_cos': 0.9876580660849721, 'analogy_dist': 0.4768739342689514}


In [33]:
visualize_3d_projection(
    gender_pairs=gender_clean, royalty_pairs=royalty_clean,
    embeddings=embeddings, projector=model,
    output_html='../results/linear/linear_3d.html',
    title='Linear Projection - 3D Visualization'
)

Сохранено: ../results/linear/linear_3d.html


# Внедрение FCA (join and meet)
